# fexp_panel → Parquet

Read Barra GEM4US factor exposure panel from SAS v9, keep US / USMC names (`aci_us=1` or `aci_usmc=1`), write filtered rows to `./parquet_files/`.

**Source:** `Y:\sasdata\barra\gem4us\fexp_panel.sas7bdat`  
**Output:** `./parquet_files/fexp_panel.parquet`

SAS does not support predicate pushdown. We read in row windows with `pyreadstat` (`output_format="polars"`) and filter each chunk before accumulating — no pandas involved.

In [1]:
from pathlib import Path

import polars as pl
import pyreadstat

SAS_PATH = Path(r"Y:\sasdata\barra\gem4us\fexp_panel.sas7bdat")
OUTPUT_DIR = Path("parquet_files")
OUTPUT_PATH = OUTPUT_DIR / "fexp_panel.parquet"
CHUNK_SIZE = 500_000

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
def load_filtered_fexp_panel(
    sas_path: Path,
    chunk_size: int = CHUNK_SIZE,
) -> pl.DataFrame:
    """Scan SAS file in chunks; keep rows where aci_us=1 or aci_usmc=1."""
    filtered_chunks: list[pl.DataFrame] = []
    rows_read = 0
    rows_kept = 0

    sas_path_str = str(sas_path)
    _, meta = pyreadstat.read_sas7bdat(
        sas_path_str,
        metadataonly=True,
        output_format="polars",
    )
    num_rows = meta.number_rows

    for offset in range(0, num_rows, chunk_size):
        chunk, _meta = pyreadstat.read_sas7bdat(
            sas_path_str,
            row_offset=offset,
            row_limit=chunk_size,
            output_format="polars",
        )
        if chunk.height == 0:
            break

        rows_read += chunk.height
        kept = chunk.filter((pl.col("aci_us") == 1) | (pl.col("aci_usmc") == 1))
        if kept.height:
            filtered_chunks.append(kept)
            rows_kept += kept.height

    if not filtered_chunks:
        return pl.DataFrame()

    result = pl.concat(filtered_chunks, how="vertical_relaxed")
    print(f"Read {rows_read:,} rows; kept {rows_kept:,} ({rows_kept / rows_read:.1%})")
    return result

In [5]:
if not SAS_PATH.exists():
    raise FileNotFoundError(f"SAS dataset not found: {SAS_PATH}")

fexp_panel = load_filtered_fexp_panel(SAS_PATH)
fexp_panel.write_parquet(OUTPUT_PATH)

print(f"Wrote {fexp_panel.height:,} rows to {OUTPUT_PATH.resolve()}")
fexp_panel.head()

Read 6,997,823 rows; kept 1,238,718 (17.7%)
Wrote 1,238,718 rows to C:\Users\algertp\cursorprojects\plfrets\parquet_files\fexp_panel.parquet


barrid,yyyymm,date,name,cusip,sedol,localid,country,country_gem4,WORLD,BETA,BTOP,DIVYILD,EARNQLTY,EARNVAR,EARNYILD,GROWTH,INVSQLTY,LEVERAGE,LIQUIDTY,LTREVRSL,MIDCAP,MOMENTUM,PROFIT,RESVOL,SIZE,AEROSPCE,AIRLINES,DIVMETAL,AUTOCOMP,BANKS,BIOTECH,BLDCNSTR,CHEMICAL,COMMSVCS,COMMUNIC,COMPUTER,…,PHARMA,PRECMETL,REALEST,RETAIL,SEMICOND,SMICNDEQ,STEEL,TELECOM,TRNSPORT,UTILITY,CAPMRKTS,RGNLBNKS,THRIFTS,RLESTMNG,ret,indname,aci_us,aci_usmc,aci_gnp,aci_sci,aci_lqa1,aci_lmc,yield,trisk,srisk,hbeta,pbeta,currency,price_loc,capt_loc,price_usd,capt_usd,glbeta,aci_sc,aci_sci2,aci_sc2,aci_usmc2
str,f64,date,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""USAA181""",199501.0,1994-12-31,"""AAON INC""","""000360206""","""2268130""","""USAAON""","""USA""","""USA""",1.0,-0.02,0.725,-1.205,-0.038,1.069,0.206,1.718,-0.399,-1.085,1.613,-0.562,-2.694,0.979,-0.606,0.946,-3.161,0.0,0.0,0.0,0.0,0.0,0.0,0.946,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.02913,"""BLDCNSTR""",0.0,1.0,null,null,null,null,0.0,55.93577,54.265051,0.998257,1.16788,"""USD""",12.875,7.1391875e7,12.875,7.1391875e7,null,null,null,null,null
"""USAA191""",199501.0,1994-12-31,"""AAR CORP""","""000361105""","""2001119""","""USAIR""","""USA""","""USA""",1.0,-0.665,2.187,0.492,-0.513,-0.216,-0.116,-0.524,0.99,0.546,-0.723,-0.133,-0.534,-0.334,-1.04,-0.843,-2.418,0.721,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.01832,"""AEROSPCE""",1.0,0.0,null,null,null,null,3.586303,22.273257,20.551084,0.678909,0.661294,"""USD""",13.375,2.12756125e8,13.375,2.12756125e8,null,null,null,null,null
"""USAA1I1""",199501.0,1994-12-31,"""AG-BAG INTL LTD""","""001077106""","""2929620""","""USAGBG""","""USA""","""USA""",1.0,-1.712,1.59,-1.205,0.536,1.471,-0.218,0.067,0.015,-1.085,1.649,0.402,-2.716,0.666,-0.928,1.716,-4.483,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2381,"""CONSTPP""",0.0,1.0,null,null,null,null,0.0,66.922507,65.561704,0.163032,1.054802,"""USD""",1.3125,1.176525e7,1.3125,1.176525e7,null,null,null,null,null
"""USAA1W1""",199501.0,1994-12-31,"""ACS ENTERPRISES INC""","""000872309""","""2001302""","""USACSE""","""USA""","""USA""",1.0,0.583,0.131,-1.205,0.326,1.427,-2.842,0.981,0.079,-1.085,1.191,-1.518,-2.692,-1.918,-0.403,2.176,-3.043,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.34286,"""MEDIA""",0.0,1.0,null,null,null,null,0.0,60.512611,58.344472,1.293973,1.371453,"""USD""",8.75,9.121875e7,8.75,9.121875e7,null,null,null,null,null
"""USAA1Y1""",199501.0,1994-12-31,"""A D C TELECOMMUNICATN""","""000886101""","""2554635""","""USADCT""","""USA""","""USA""",1.0,1.361,-0.998,-1.205,-0.157,-0.35,-0.811,0.877,-0.318,-1.16,1.124,-0.99,0.869,0.527,1.503,0.606,-1.02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.206,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.02,"""COMMUNIC""",1.0,0.0,null,null,null,null,0.0,37.542071,31.496547,1.677502,1.684321,"""USD""",50.0,1.3950e9,50.0,1.3950e9,null,null,null,null,null


In [6]:
# Round-trip check
pl.scan_parquet(OUTPUT_PATH).select(pl.len()).collect()

len
u32
1238718
